In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel
import torch
import re
from tqdm import tqdm

import plotly.express as px
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings("ignore")

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("✅ GPU kullanılıyor")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("⚠️ GPU yok, CPU kullanılıyor")

In [ ]:
df = pd.read_csv("articles_clean.csv")
df.head()

In [ ]:
model_name = 'dbmdz/bert-base-turkish-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

print(f"✅ Model loaded: {model_name}")

In [ ]:
# 1. Ağırlıkları Belirliyoruz
W_TITLE = 0.20
W_ABSTRACT = 0.70
W_KEYWORDS = 0.10

def get_berturk_embeddings(text):
    """
    Tek bir metin için BERTurk embedding üretir.
    Boş veri gelirse BERTurk'e uygun 768 boyutlu sıfır vektörü döner.
    """
    if not isinstance(text, str) or text.strip() == "":
        return np.zeros(768, dtype=np.float32)

    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512, padding=True).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy().squeeze()
    return embedding

# Pandas işlemlerine görsel ilerleme çubuğu (tqdm) ekler
tqdm.pandas()
print("BERTurk ayrı alan embedding süreci başlatılıyor...")

# 2. Her alan için ayrı ayrı vektör üretiyoruz
df["title_embedding"] = df["title_tr_clean"].progress_apply(get_berturk_embeddings)
df["abstract_embedding"] = df["abstract_tr_clean"].progress_apply(get_berturk_embeddings)
df["keywords_embedding"] = df["keywords_tr_clean"].progress_apply(get_berturk_embeddings)

def combine_embeddings(row):
    """3 ayrı vektörü ağırlıklarına göre çarpar, toplar ve normalize eder."""
    v_title = np.array(row["title_embedding"])
    v_abstract = np.array(row["abstract_embedding"])
    v_keywords = np.array(row["keywords_embedding"])

    # Ağırlıklı Toplam: w1*v1 + w2*v2 + w3*v3
    weighted_vector = (W_TITLE * v_title) + (W_ABSTRACT * v_abstract) + (W_KEYWORDS * v_keywords)

    # L2 Normalizasyon
    norm = np.linalg.norm(weighted_vector)
    if norm > 0:
        weighted_vector = weighted_vector / norm

    return weighted_vector.tolist()

print("\nAğırlıklı vektörler birleştiriliyor ve normalize ediliyor...")
# 3. Vektörleri birleştirip nihai "embedding" sütununu oluşturuyoruz
df["embedding"] = df.apply(combine_embeddings, axis=1)

print("\nAğırlıklı Embedding işlemi başarıyla tamamlandı.")
print(f"Toplam Kayıt: {len(df)}")
print(f"Yeni Vektör Boyutu: {len(df['embedding'].iloc[0])}")

df.drop(columns=["title_embedding", "abstract_embedding", "keywords_embedding"], inplace=True)

In [ ]:
df.head()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display
import re

# 1. Temizleme Fonksiyonu
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"\s+", " ", text)          # Fazla boşluklar
    text = re.sub(r"[#%&*_=+<>]", "", text)   # Anlamsız özel karakterler
    return text.strip()

# 2. Örnek girdi
raw_query_title = "Hava Muharebesinde Otonom Savunma Algoritmasının Geliştirilmesi"

raw_query_abstract = (
    "Bu çalışma kapsamında, temel hava muharebesi manevraları kullanılarak "
    "birebir muharebeler için otonom savunma algoritması geliştirilmiştir. "
    "Algoritma, hedef hava aracı ile beklenmedik bir şekilde karşılaşıldığı durumlarda "
    "saldırı üstünlüğünün sağlanması için en uygun muharebe manevrasını seçmeyi sağlamaktadır. "
    "Algoritmanın test edilmesi amacıyla saldıran ve savunan uçaklar için doğrusal olmayan "
    "dinamik modeller kullanılmıştır. Algoritmada manevra seçimi için temel savaş "
    "manevralarını içeren manevra kütüphanesinden uygun manevrayı seçecek kural tabanlı "
    "bir yapı önerilmiştir. MATLAB/Simulink ortamında yapılan benzetim çalışmaları ile "
    "algoritmanın başarımı test edilmiş ve sonuçlar gösterilmiştir."
)

raw_query_keywords = "bire-bir hava muharebesi, kural tabanlı yöntem, temel hava muharebe manevraları."

# 3. Verileri temizliyoruz
query_title = clean_text(raw_query_title)
query_abstract = clean_text(raw_query_abstract)
query_keywords = clean_text(raw_query_keywords)

# 4. SORGUDAN AĞIRLIKLI EMBEDDING ÜRETİMİ (Yeni Mantık)
# Ayrı ayrı vektörlere çeviriyoruz
v_title = get_berturk_embeddings(query_title)
v_abstract = get_berturk_embeddings(query_abstract)
v_keywords = get_berturk_embeddings(query_keywords)

# Veritabanında kullandığımız aynı ağırlıklarla çarpıp topluyoruz
weighted_query_vector = (W_TITLE * v_title) + (W_ABSTRACT * v_abstract) + (W_KEYWORDS * v_keywords)

# L2 Normalizasyonu (Sorguyu da diğerleri gibi normalize ediyoruz)
norm = np.linalg.norm(weighted_query_vector)
if norm > 0:
    weighted_query_vector = weighted_query_vector / norm

# Sklearn'in cosine_similarity fonksiyonu için 2 boyutlu hale getiriyoruz
query_vector = weighted_query_vector.reshape(1, -1)

# 5. Karşılaştırma
corpus_embeddings = np.array(df["embedding"].tolist())
similarity_scores = cosine_similarity(query_vector, corpus_embeddings)[0]

# 6. Sonuçları raporlama
df["similarity_score"] = similarity_scores
top_results = df.sort_values(by="similarity_score", ascending=False)

print("🎯 En Benzer Sonuçlar (Top 5):")
with pd.option_context('display.max_colwidth', None):
    # Tablonuzdaki orijinal sütun isimlerine göre ("Title_TR", "title_tr_clean" vb.) burayı ayarlayabilirsiniz.
    display(top_results[["Year", "title_tr_clean", "abstract_tr_clean", "similarity_score"]].head(5))

print("\n----------------------------------------------------------------------\n")

print("🛑 En Az Benzer Sonuçlar (Bottom 5):")
with pd.option_context('display.max_colwidth', None):
    display(top_results[["Year", "title_tr_clean", "abstract_tr_clean", "similarity_score"]].tail(5))

In [ ]:
# 1. Veri Hazırlığı
embeddings = np.array(df['embedding'].tolist())

# 2. t-SNE ve Kümeleme
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
embeddings_2d = tsne.fit_transform(embeddings)

kmeans = KMeans(n_clusters=6, random_state=42)
df['cluster'] = kmeans.fit_predict(embeddings)

# Grafik koordinatlarını ekle
df['x'] = embeddings_2d[:, 0]
df['y'] = embeddings_2d[:, 1]

# 3. Plotly Etkileşimli Grafiği Oluşturma
fig = px.scatter(
    df,
    x='x',
    y='y',
    color='cluster',
    hover_name='Title_TR',
    hover_data={
        'x': False,
        'y': False,
        'Year': True,
        'cluster': True
    },
    title='TUSAŞ LIFT UP Projeleri Anlamsal Benzerlik Uzayı (Etkileşimli t-SNE) - BERTurk',
    labels={'cluster': 'Küme No', 'Year': 'Yıl'},
    template='plotly_white',
    color_continuous_scale='Viridis'
)

# Grafik ayarlarını güncelle
fig.update_traces(marker=dict(size=10, opacity=0.7, line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(
    dragmode='pan',
    hoverlabel=dict(bgcolor="white", font_size=12, font_family="Arial")
)

fig.show()